In [2]:
# | eval: false

In [3]:
#| default_exp diffusion_utils
import sys
sys.path.append("..")

In [4]:
#| export
import numpy as np
import torch 
import torch.nn as nn
from ddpm_backtest.models import FiLMLayer,f_net
from ddpm_backtest.noising_time import cosine_beta_scheduler,timestep_embedding

Device: cpu | T=100


In [5]:
#| export
class TabularDDPM(nn.Module):
    def __init__(self, d_in, cond_in_classes, scaled_cont_dim,
                 fixed_market_dim, time_dim, t_dim=64, dropout=0.1):
        super().__init__()
        self.t_dim = t_dim
        self.cond_emb = nn.Embedding(cond_in_classes + 1, t_dim, padding_idx=0)
        market_in = scaled_cont_dim + fixed_market_dim
        self.market_encoder = nn.Sequential(
            nn.Linear(market_in, t_dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(t_dim, t_dim),     nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        self.time_encoder = nn.Sequential(
            nn.Linear(time_dim, t_dim // 2), nn.SiLU(),
            nn.Linear(t_dim // 2, t_dim // 2),
        )
        self.cond_fuse = nn.Sequential(
            nn.Linear(t_dim + t_dim // 2, t_dim), nn.SiLU(),
        )
        H = 128
        self.fc1    = nn.Linear(d_in + t_dim + t_dim, H)
        self.fc2    = nn.Linear(H, H)
        self.fc3    = nn.Linear(H, H)
        self.fc_out = nn.Linear(H, d_in)
        self.act    = nn.SiLU()
        self.drop   = nn.Dropout(dropout)
        self.film1  = FiLMLayer(H, t_dim)
        self.film2  = FiLMLayer(H, t_dim)
        self.film3  = FiLMLayer(H, t_dim)

    def forward(self, x, c, market_cont, time_feats, t, drop_mask=None):
        t_emb = timestep_embedding(t, dim=self.t_dim)
        if drop_mask is not None and drop_mask.any():
            market_in = market_cont.clone(); market_in[drop_mask] = 0.0
            c_eff     = c.clone();           c_eff[drop_mask]     = 0
        else:
            market_in, c_eff = market_cont, c
        c_emb    = self.cond_emb(c_eff).to(x.device)
        cond_emb = self.cond_fuse(torch.cat([
            self.market_encoder(market_in),
            self.time_encoder(time_feats),
        ], dim=-1))
        h = torch.cat([x, c_emb, t_emb], dim=1)
        h = self.act(self.fc1(h));            h = self.film1(h, cond_emb)
        h = self.drop(self.act(self.fc2(h))); h = self.film2(h, cond_emb)
        h = self.drop(self.act(self.fc3(h))); h = self.film3(h, cond_emb)
        return self.fc_out(h)

In [6]:
#| export
class EMA:
    def __init__(self,model,decay=0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k,v in model.named_parameters()}
    @torch.no_grad()
    def update(self,model):
        for k,v in model.named_parameters():
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v
    def apply(self,model):
        self.backup = {k: v.clone() for k,v in model.named_parameters()}
        for k,p in model.named_parameters():p.data.copy_(self.shadow[k])
    def restore(self,model):
        for k,p in model.named_parameters():p.data.copy_(self.backup[k])